In [6]:
import numpy as  np
import pandas as pd

# Import functions for interpolation
from scipy.interpolate import RegularGridInterpolator 
from scipy.optimize import minimize

In [7]:
# Set up to read data from both master phtometry and master target lists
def read_star_row_from_csv(
    host_name: str,
    sigma_mag: bool = True,
    sigma_parallax: float = 0.1,
):
    master_csv_path = "ASTR502_Mega_Target_List_1.csv"
    master_phot_csv_path = "ASTR502_Master_Photometry_List_1.csv"
    
    master_df = pd.read_csv(master_csv_path)
    phot_df = pd.read_csv(master_phot_csv_path)
    index = master_df[master_df["hostname"] == host_name].index[0]
    phot_index = phot_df[phot_df["hostname"] == host_name].index[0]
    # Select row
    row = master_df.iloc[index]
    phot_row = phot_df.iloc[phot_index]

    # Build props dict
    props = {}

    # --- Parallax (required) ---
    # Need this for magnitudes to mean anything. 
    if not np.isnan(row["st_parallax_mas"]):
        props["parallax"] = (row["st_parallax_mas"], sigma_parallax)

    # band map from master_phot_csv to MIST keys
    band_map = {
        "gaiaGmag": "G_mag",
        "giaaBPmag": "BP_mag",
        "gaiaRPmag": "RP_mag",
        "Jmag": "J_mag",
        "Hmag": "H_mag",
        "Kmag": "K_mag",
    }
    #get photometry from phot_df
    for csv_col, iso_key in band_map.items():
        if csv_col in phot_row and not np.isnan(phot_row[csv_col]):
            sigmamag = 0.02 if sigma_mag is False else phot_row[f"e_{csv_col}"]
            props[iso_key] = phot_row[csv_col]
            props[f"{iso_key}_err"] = sigmamag


    return props

In [8]:
# Load in the data using read_star_row_from_csv
sun = read_star_row_from_csv("HD 209458")
# Checking the data to make sure it was collected properly
print(sun)

{'parallax': (np.float64(20.7693898852824), 0.1), 'G_mag': np.float64(7.521244378), 'G_mag_err': np.float64(0.0183144879635296), 'RP_mag': np.float64(7.080287415), 'RP_mag_err': np.float64(0.0264984319384077), 'J_mag': np.float64(6.591), 'J_mag_err': np.float64(0.02), 'H_mag': np.float64(6.366), 'H_mag_err': np.float64(0.038), 'K_mag': np.float64(6.308), 'K_mag_err': np.float64(0.026)}


In [ ]:
# Define the Chi-Squared Function
def objective_function(params, observed_props):
    """
    params: [mass, log_age, metallicity]
    observed_props: the dictionary returned by your read_star_row_from_csv
    """
    mass, log_age, feh = params
    
    # Get predicted magnitudes from your interpolator
    # Note: You must have an interpolator object 'iso_interp' defined globally or passed in
    try:
        predicted_mags = iso_interp([mass, log_age, feh]) 
    except:
        return 1e10 # Return a huge penalty if params are outside grid bounds

    chi_sq = 0
    
    # Compare predicted Absolute Magnitudes to Observed Absolute Magnitudes
    # Distance Modulus: μ = 5 * log10(d_pc) - 5
    parallax_mas = observed_props['parallax'][0]
    dist_pc = 1000.0 / parallax_mas
    dist_mod = 5 * np.log10(dist_pc) - 5

    for band in ["G_mag", "BP_mag", "RP_mag"]:
        if band in observed_props:
            obs_mag = observed_props[band]
            obs_err = observed_props[f"{band}_err"]
            
            # Predict apparent mag: Apparent = Absolute + Distance Modulus
            pred_mag = predicted_mags[band] + dist_mod
            
            chi_sq += ((obs_mag - pred_mag) / obs_err)**2

    return chi_sq

In [ ]:
# Optimization for age estimation
def estimate_age(star_name, initial_guess, bounds):
    # Get data
    star_data = read_star_row_from_csv(star_name)
    
    # Initial guess: [Mass, Log10(Age), [Fe/H]]
    # Define bounds (Mass, Age, FeH)
    
    result = minimize(
        objective_function, 
        initial_guess, 
        args=(star_data,), 
        method='L-BFGS-B', 
        bounds=bounds
    )
    
    return result

In [ ]:
# For the Sun

In [ ]:
# For HD 209458
HD_initial_guess = [1.0, 9.5, 0.0]
HD_bounds = [(0.1, 5.0), (7.0, 10.1), (-2.0, 0.5)]

hd_result = estimate_age("HD 209458", HD_initial_guess, HD_bounds)
print(f"Results for HD 209458: {hd_result.x}")
print(f"Estimated Age (Gyr): {10**(hd_result.x[1]) / 1e9:.2f}")

Results for HD 209458: [1.  9.5 0. ]
Estimated Age (Gyr): 3.16


In [13]:
# For HIP 8152
HIP_initial_guess = [0.937, 9.78, 0.007]
HIP_bounds = [(0.88, 1.00), (9.0, 10.1), (-0.15, 0.15)]

HIP_result = estimate_age("HIP 8152", HIP_initial_guess, HIP_bounds)
print(f"Results for HD 209458: {HIP_result.x}")
print(f"Estimated Age (Gyr): {10**(HIP_result.x[1]) / 1e9:.2f}")

Results for HD 209458: [9.37e-01 9.78e+00 7.00e-03]
Estimated Age (Gyr): 6.03


In [12]:
# Test minimize to give back how it is choosing.

